# 🔬 Google Colab GPU Runner: Generative Synthetic Augmentation (S1–S3, Few-Shot & Final Combined)
**Research Goal:** Test whether Stable Diffusion 1.5 + LoRA synthetic images improve minority recall over strong loss baselines.

**Leakage Safeguard:** Generators train ONLY on training-fold real `df` samples. Memorization filter purges samples with DINOv2 cosine similarity $\ge 0.95$.


In [ ]:
# 1. Verify GPU Acceleration
import torch
assert torch.cuda.is_available(), 'GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.'
print(f'Active GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Mount Drive & Setup Environment
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/rare_class_benchmark'
os.makedirs(DRIVE_DIR, exist_ok=True)
if os.path.exists('/content/rare-class-benchmark'):
    %cd /content/rare-class-benchmark
elif os.path.exists('scripts/run_all.py'):
    pass
elif os.path.exists('/content/scripts/run_all.py'):
    %cd /content
else:
    !git clone https://github.com/dhina-research/rare-class-skin-lesion-benchmark.git /content/rare-class-benchmark 2>/dev/null || true
    if os.path.exists('/content/rare-class-benchmark'):
        %cd /content/rare-class-benchmark
!pip install -q diffusers peft accelerate transformers timm scikit-learn pandas matplotlib seaborn pyyaml tqdm


In [ ]:
# 3. Execute Generative Arms (S1 MixUp, S2 SD1.5+LoRA @ 50%, 100%, 200%, Final Combined)
!python scripts/run_all.py --arms S1 S2_50 S2_100 S2_200 FINAL_COMBINED --seeds 42 1337 2024 --folds 0 1 2 3 4


In [ ]:
# 4. Generate Blind Real-vs-Synthetic Grid & Saliency Explainability Panels
from src.synth import export_blind_real_vs_synthetic_grid
print('Exporting diagnostic quality grid...')
!python -c "import pandas as pd; from src.stats import aggregate_benchmark_runs; aggregate_benchmark_runs()"


In [ ]:
# 5. Final Paired Significance Testing: Does Synthetic Augmentation Beat Focal Loss?
import pandas as pd
from src.stats import compute_paired_comparison, aggregate_benchmark_runs
df_summary, df_runs = aggregate_benchmark_runs('results/logs/runs.csv')

# Paired comparison: SD1.5 100% vs Focal Loss B4
b4_vals = df_runs[df_runs['arm_id'] == 'B4']['test_minority_recall'].values
s2_vals = df_runs[df_runs['arm_id'] == 'S2_100']['test_minority_recall'].values
if len(b4_vals) > 0 and len(s2_vals) > 0:
    res = compute_paired_comparison(b4_vals, s2_vals, metric_name='minority_recall')
    print('Paired Hypothesis Test (S2_100 vs B4 Focal Loss):')
    print(f"  Delta Mean: {res['delta_mean']:.4f}")
    print(f"  95% CI: [{res['delta_ci_95'][0]:.4f}, {res['delta_ci_95'][1]:.4f}]")
    print(f"  Wilcoxon p-value: {res['p_value']:.4f}")
    print(f"  Conclusion: {res['conclusion']}")

# Sync final results
!cp -r results/ {DRIVE_DIR}/ 2>/dev/null || true
!zip -r /content/complete_benchmark_results.zip results/
from google.colab import files
files.download('/content/complete_benchmark_results.zip')
